# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: FlyRank, *The State of AI-Driven SEO*, March 2026 -- 341,701 content pieces, 57 brands.

### Finding #2 — "The Content Performance Curve"

**Claim:** health score peaks at 61-90 days (33.1), plateaus through 91-180, then hits a "decay cliff" at 271-365 days (14.0), described as a lifecycle a page moves through.

- **Where does the label come from?** A single cross-sectional snapshot: every content-age bucket is a *different set of pages*, each measured once, as of March 2026. No page is tracked across its own 90 -> 270 -> 365 day journey.
- **Does the validation design carry the claim?** Only partly. "Pages currently 271-365 days old score lower than pages currently 61-90 days old" is well-supported. "A page will hit a decay cliff around month 9" is a stronger, longitudinal claim the design doesn't test — the 271-365-day cohort was *published* roughly 9-12 months before the snapshot, in a different competitive and algorithmic moment than the 61-90-day cohort. Cohort composition and age are confounded here in exactly the way the paper is careful about elsewhere (it explicitly flags survivor bias for the `365+ x 361+` cell in Finding #8, but not this one).
- **My question, constructively:** has this pattern been checked on a *panel* — the same pages' health scores re-measured as they individually cross the 90/180/270/365-day marks — rather than different pages compared at one point in time? That would separate "aging causes decline" from "the pages that happen to be 9+ months old right now were a weaker cohort to begin with," and would make the recommended 6-9 month review-cycle timing a stronger, individually-grounded claim rather than a portfolio-average one.

### Finding #4 — "The Freshness Multiplier" (refresh boost)

**Claim:** 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions than similarly old, unrefreshed content; the playbook calls refresh timing "one of the strongest measured levers available."

- **Where does the label come from?** "Refreshed" vs. "not refreshed" is which freshness bucket a page falls into — a bucket set by whether an editor *chose* to update that page recently, not by random assignment.
- **Does the validation design carry the claim?** This is an observational comparison between a self-selected treatment group and everyone else, not a matched or randomized comparison. Editors plausibly refresh pages that already show promise — existing backlinks, past rankings, brand priority — so part of the 3.2x/57x gap could be *which pages get chosen* for refresh work rather than *what refreshing does* to a page. The paper is careful about this exact selection-bias pattern elsewhere (Finding #8 explicitly warns the small `365+ x 361+` survivor cell "should not be used as a headline decay proof point"), which makes the lever/causal framing here worth the same caution.
- **My question, constructively:** is there a matched comparison available — refreshed pages vs. similarly-aged, similarly-authoritative pages that were *not* refreshed, matched on pre-refresh health/impressions — or a staggered-rollout angle, that would help separate "refreshing helps" from "the pages worth refreshing were already positioned to recover on their own"?

In [ ]:
print('Finding #2 (Content Performance Curve): cross-sectional age buckets vs a true panel')
print('  question is whether the same pages were tracked through their own lifecycle, or')
print('  different pages at different ages were compared once.')
print()
print('Finding #4 (Freshness Multiplier / refresh boost): refreshed vs unrefreshed is a')
print('  self-selected group, not randomized -- question is whether a matched comparison')
print('  exists to separate the refresh effect from which pages get chosen to refresh.')

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** plain `StratifiedKFold`, ignoring which client each page belongs to — rows from the same client can land in both train and test. **After:** `GroupKFold` on `client_hash_id`, the split already used for the real Week-5 numbers. Same frame, same model, same metric — the only thing that changes is whether client identity leaks across the split.

In [ ]:
%pip -q install duckdb scikit-learn
import os, getpass
import numpy as np
import pandas as pd
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = 'hf://datasets/FlyRank/internship-warehouse'
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"
DECISION_DATE = pd.Timestamp('2026-03-31')
SEED = 42

# Identical frame + features to w05_model.ipynb.
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS mar_impr, SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*) AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.mar_impr, m.mar_clicks, m.mar_days,
       m.mar_impr * 1.0 / m.mar_days AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0) AS mar_avg_position,
       m.mar_clicks * 100.0 / NULLIF(m.mar_impr, 0) AS mar_ctr,
       (m.h2_impr - m.h1_impr) * 1.0 / NULLIF(m.h2_impr + m.h1_impr, 0) AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0) AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame['mar_h2_vs_h1'] = frame['mar_h2_vs_h1'].fillna(0.0)
frame['apr_daily_impr'] = frame['apr_daily_impr_raw'].fillna(0.0)
frame['y'] = (frame['apr_daily_impr'] < 0.80 * frame['mar_daily_impr']).astype(int)
frame = frame.drop(columns=['apr_daily_impr_raw'])

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_updated_date FROM {DIM_CONTENT}").df()
frame = frame.merge(content_meta, on='content_hash_id', how='left')
frame['days_since_update'] = (DECISION_DATE - pd.to_datetime(frame['content_updated_date'])).dt.days
frame['days_since_update'] = frame['days_since_update'].fillna(frame['days_since_update'].median())
frame['content_type'] = frame['content_type'].fillna('unknown')
frame['log_daily_impr'] = np.log1p(frame['mar_daily_impr'])
type_dummies = pd.get_dummies(frame['content_type'], prefix='type', dtype=int)
frame = pd.concat([frame, type_dummies], axis=1)

FEATURE_COLS = ['log_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1', 'days_since_update'] + list(type_dummies.columns)
BASE_RATE = frame['y'].mean()
print(f"frame: {len(frame):,} pages | {frame['client_hash_id'].nunique()} clients | base rate {BASE_RATE:.3f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def cv_precision(cv_iter, K_VALUES=(10, 20, 50)):
    scores = {k: [] for k in K_VALUES}
    for tr, te in cv_iter:
        X_tr, X_te = frame.iloc[tr][FEATURE_COLS], frame.iloc[te][FEATURE_COLS]
        y_tr, y_te = frame.iloc[tr]['y'], frame.iloc[te]['y']
        rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                     class_weight='balanced', random_state=SEED, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        proba = rf.predict_proba(X_te)[:, 1]
        for k in K_VALUES:
            scores[k].append(precision_at_k(proba, y_te.to_numpy(), k))
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in scores.items()}

# ---- BEFORE: random split, client identity ignored ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
before = cv_precision(skf.split(frame, frame['y']))

# ---- AFTER: grouped by client, the Week-5 split ----
gkf = GroupKFold(n_splits=5)
after = cv_precision(gkf.split(frame, groups=frame['client_hash_id']))

print(f"{'K':>6} {'BEFORE (random)':>18} {'AFTER (grouped)':>18} {'gap':>8}")
for k in (10, 20, 50):
    b_mean, b_std = before[k]
    a_mean, a_std = after[k]
    print(f"{k:>6} {b_mean:>12.3f}(+/-{b_std:.3f}) {a_mean:>12.3f}(+/-{a_std:.3f}) {b_mean - a_mean:>+8.3f}")

p50_before, p50_after = before[50][0], after[50][0]
print(f"\nIf BEFORE > AFTER at p@50: part of last week's headline number was the random split")
print(f"letting the model see other pages from the same client during training, not real signal.")


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Test 1: no label-derived column is anywhere in the final feature set
LABEL_DERIVED_NAMES = ['apr', 'y', 'is_down', 'baseline_score']
suspect = [c for c in FEATURE_COLS if any(bad in c.lower() for bad in LABEL_DERIVED_NAMES)]
print(f"Feature names matching a label-derived pattern: {suspect or 'none'}")
assert not suspect, 'a label-derived column name is sitting in the final feature set'

# Test 2: per-feature correlation with the label (nothing should sit near 1.0 )
corr = {c: abs(np.corrcoef(frame[c], frame['y'])[0, 1]) for c in FEATURE_COLS if frame[c].std() > 0}
print('\n|correlation| with the April label, final feature set:')
for c, v in sorted(corr.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {c:20s} {v:.4f}")
assert max(corr.values()) < 0.90, 'a feature sits suspiciously close to the label'
print('max |r| below 0.90                                          PASS')


In [ ]:
# Test 3: is days_since_update a clean as-of snapshot? (repeat of the ML-05 check)
future_updates = (pd.to_datetime(frame['content_updated_date']) > DECISION_DATE).sum()
print(f"content_updated_date AFTER the decision date: {future_updates} of {len(frame):,} rows")
print('Still a named limitation if nonzero -- carried forward from ML-05, not newly discovered here.')

# Test 4: grouped-vs-random gap, restated from Section 2 as the leakage finding it is
print(f"\nRandom-split precision@50: {p50_before:.3f}")
print(f"Grouped-split precision@50: {p50_after:.3f}")
print(f"Gap: {p50_before - p50_after:+.3f} -- this IS the leakage audit's headline number for this notebook.")


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Two claims worth tightening from my own Week 4-5 write-ups, using the claim ladder: observed → directional → decision-support, never causal without an experiment I didn't run.

In [ ]:
n_test_50 = min(50, len(frame) // 5)

before_claim_1 = (
 "BEFORE: 'My Random Forest predicts which pages will decline in April, achieving precision@50 "
 "far above the baseline -- this proves the momentum and staleness signals drive content decline.'"
)
after_claim_1 = (
 f"AFTER: 'Ranking by this model's score on a client-grouped held-out slice, the top 50 pages "
 f"contained {p50_after:.0%} true April declines versus a {BASE_RATE:.0%} base rate (n={len(frame):,} "
 f"pages, {frame['client_hash_id'].nunique()} clients). That is an observed, decision-support ranking "
 f"on this dataset and period -- not a causal claim about what makes a page decline, and not yet "
 f"validated on any month outside March-April 2026.'"
)

before_claim_2 = (
 "BEFORE (from w04_signal_audit.ipynb): 'A CONFIRMED staleness signal means the existing "
 "refresh-flag logic is doing real work.'"
)
after_claim_2 = (
 "AFTER: 'In the March 2026 slice audited in w04_signal_audit.ipynb, pages in the stalest "
 "quintile showed a directionally higher April decline rate than the freshest quintile, above "
 "the n=50 floor. This is associational, on one month of one warehouse -- it supports using "
 "staleness as one input for prioritization, not a claim that refreshing a page causes it to "
 "stop declining.'"
)

for b, a in [(before_claim_1, after_claim_1), (before_claim_2, after_claim_2)]:
    print(b)
    print(a)
    print()

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.